### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [3]:
## Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [4]:
## Step 2: FAISS VectorStore with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

d:\Mission Project\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4736.16it/s]


In [5]:
## Step 3: Create MMR Retriever
retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs={"k":3}
)

In [13]:
## Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context: 
{context}

Question: {input}
""")
llm = ChatGroq(model = "openai/gpt-oss-120b")

In [14]:
## Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [15]:
## Step 6: Query
query = {"input": "How does Langchain support agents and memory?"}
response = rag_chain.invoke(query)

print("Answer:\n", response["answer"])

Answer:
 **LangChain’s support for agents and memory**

| Aspect | What LangChain provides | How it works |
|--------|------------------------|--------------|
| **Agents** | • A generic **Agent** class that can reason about which tool to use next.<br>• Built‑in tool integrations (e.g., calculators, search APIs, custom functions, database queries).<br>• Ability to call **external APIs** and databases from within a single LLM‑driven workflow. | 1. The LLM receives a prompt that includes a list of available tools.<br>2. The LLM “decides” (via its output) which tool to invoke and with what arguments.<br>3. The LangChain agent parses that decision, calls the selected tool, captures the result, and feeds it back to the LLM.<br>4. This loop repeats until the agent signals completion, enabling multi‑step, tool‑driven reasoning. |
| **Memory** | • **ConversationBufferMemory** – stores the raw dialogue history (user ↔ LLM) and can prepend it to each new prompt.<br>• **ConversationSummaryMemory**

In [16]:
response

{'input': 'How does Langchain support agents and memory?',
 'context': [Document(id='7f19bc0e-07ef-461f-9ca7-feacc345bb29', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='4d589fe7-9877-460a-8e1c-b88cb3b9f4b7', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='8c681806-d7f3-41e5-9b90-bbca1464592f', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order duri